In [ ]:
import pandas as pd
import xarray as xr

import sys
sys.path.append("../src")

from yield_spei_perturbation_simulation import add_year_month, add_spei
from spatiotemporal_yield_loss_estimation import *

In [ ]:
vulnerability_params_df = pd.read_csv(
    "../outputs/3_vuln_curve_parameters/"
    "crop_country_vulnerability_curve_parameters.csv"
)

In [ ]:
yield_fao_df = pd.read_csv(
    "../data/yield_fao-owid_adm0_processed/yield_rubber.csv"
)

crop_name = "rubber"

## run case per country

In [ ]:
country_code = "vn"

grid_df = pd.read_csv(
    "../outputs/3.2_simulated_yield/c_rubber/"
    "rubber_grid_yield_vn_beta0p3_260720.csv"
)

grid_yield_output_path = (
    f"../outputs/4.1_spatiotemporal_yield_loss/"
    f"grid_yield_rubber_extended_period_{country_code}.csv"
)
country_yield_output_path = (
    f"../outputs/4.1_spatiotemporal_yield_loss/"
    f"country_yield_rubber_extended_period_{country_code}.csv"
)

In [ ]:
grid_df = (
    grid_df[["GID", "centroid_x", "centroid_y", "harvest_area_ha"]]
        .drop_duplicates()
        .reset_index(drop=True)
)

ds = xr.open_dataset("../data/2_spei_0p5deg_processed/spei3_0p5deg.nc")
da = ds["SPEI3"]

grid_df = add_year_month(
    grid_df,
    sim_start_time="1991/01/01",
    sim_end_time="2020/12/31",
)

grid_df = add_spei(grid_df, da)

In [ ]:
yield_df = yield_fao_df[["Year", country_code]].copy()
yield_df.columns = ["year", "yield_t_ha"]

yield_df["yield_trend_t_ha"] = 0

yield_df["yield_detrended_t_ha"] = yield_df["yield_t_ha"] - yield_df["yield_trend_t_ha"]

yield_normal_detrended = yield_df["yield_detrended_t_ha"].mean()

print(
    f"Mean detrended yield for {country_code} {crop_name}: "
    f"{yield_normal_detrended:.2f} t/ha"
)

In [ ]:
row = vulnerability_params_df[
    (vulnerability_params_df["crop_name"]==crop_name)
    & (vulnerability_params_df["country_code"]==country_code.upper())
    ].iloc[0]

B = row.B
k = row.k
x0 = row.x0

grid_df["ylr_vc"] = logistic_ranged(grid_df["spei"], B, k, x0)
grid_df["ylr_vc"] = grid_df["ylr_vc"].fillna(0)

print(f"Max YLR by vuln curve: {grid_df["ylr_vc"].max():.3f}")

grid_df["yield_vc_t_ha"] = (1-grid_df["ylr_vc"]) * yield_normal_detrended

In [ ]:
grid_yield_df = grid_df[["GID", "year", "harvest_area_ha", "yield_vc_t_ha"]].copy()
grid_yield_df = grid_yield_df.rename(
    columns={"yield_vc_t_ha": "yield_sim_t_ha"}
)

yearly_yield_vc_df = aggregate_yearly_yield(
    grid_yield_df,
    yield_threshold=0.01*yield_normal_detrended
)

yearly_yield_vc_df = yearly_yield_vc_df.rename(
    columns={"yield_sim_t_ha": "yield_sim_vc_t_ha"}
)

yield_df = yearly_yield_vc_df.copy()

yield_df["yield_normal_t_ha"] = yield_normal_detrended
yield_df["yield_trend_t_ha"] = 0

yield_df["yield_reconstructed_vc_t_ha"] = (
    yield_df["yield_sim_vc_t_ha"]+yield_df["yield_trend_t_ha"]
)
yield_df["ylr"] = 1 -  (
    yield_df["yield_reconstructed_vc_t_ha"]/yield_df["yield_normal_t_ha"]
)

In [ ]:
grid_df.to_csv(grid_yield_output_path, index=False)
yield_df.to_csv(country_yield_output_path, index=False)